# Ensemble com Soma de Votos

Esse notebook combina as previsoes dos 5 modelos base (SVM, KNN, Extra Trees, Regressao Logistica e LightGBM) usando soma de votos por probabilidade

Ao inves de usar apenas o palpite final (0 ou 1) de cada modelo, usei a **probabilidade** que cada modelo atribuiu a classe 1. O resultado final e a media dessas probabilidades: se a media for >= 0.5, o ensemble preve 1.

## Importar bibliotecas

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

## Configuracao

- `PASTA`: defini a pasta onde estao os CSVs detalhados de predicao dos 5 modelos
- `SAIDA`: defini a pasta onde o CSV do ensemble sera salvo
- `ARQUIVOS`: criei o mapeamento de cada modelo para seu CSV detalhado
- `CHAVE_JOGO`: escolhi as colunas que, juntas, identificam um jogo unico

In [ ]:
PASTA = os.path.abspath(os.path.join('..', 'results', 'experimento_03_detalhado'))
SAIDA = os.path.abspath(os.path.join('..', 'results', 'experimento_03'))

ARQUIVOS = {
    'svm':                'svm_experimento_03_predicoes_detalhadas.csv',
    'knn':                'knn_experimento_03_predicoes_detalhadas.csv',
    'extra_trees':        'extra_trees_experimento_03_predicoes_detalhadas.csv',
    'regressao_logistica':'regressao_logistica_experimento_03_predicoes_detalhadas.csv',
    'lightgbm':           'lightgbm_experimento_03_predicoes_detalhadas.csv',
}

CHAVE_JOGO = ['Temporada', 'Data Jogo', 'Round', 'Posicao no Teste']

print('Pasta:', PASTA)
print('Saida:', SAIDA)
print('Modelos:', list(ARQUIVOS.keys()))

## Carregar os CSVs detalhados

Cada CSV contem as predicoes de um modelo. Li todos e juntei em um unico DataFrame, adicionando a coluna `_modelo` para identificar de qual algoritmo veio cada linha.

In [ ]:
dfs = []
for modelo, arquivo in ARQUIVOS.items():
    caminho = os.path.join(PASTA, arquivo)
    df = pd.read_csv(caminho)
    df['_modelo'] = modelo
    dfs.append(df)

dados = pd.concat(dfs, ignore_index=True)

print(f'Total de linhas: {len(dados)}')
print(f'Janelas disponiveis: {sorted(dados["Janela Incremental"].unique())}')
print(f'Modelos: {list(dados["_modelo"].unique())}')
dados.head()

## Calcular soma de votos por probabilidadee

Para cada janela incremental 5, 10, 15:

1. Filtrei os dados da janela e removi duplicatas
2. Criei um **pivot de probabilidades**, entã cada linha vira um jogo, cada coluna e a `Probabilidade Classe 1` de um modelo
3. Mantive apenas jogos onde **todos os 5 modelos** retornaram probabilidade
4. Calculei a **media** das probabilidades dos 5 modelos e o resultado do ensemble e `1` se a media for **>= 0.5**
5. Calculei acuracia e F1-Score por temporada

No final salvo o csv com os resultados

In [ ]:
resultados = []

for janela in [5, 10, 15]:
    dados_janela = dados[dados['Janela Incremental'] == janela].copy()

    dados_janela = dados_janela.drop_duplicates(
        subset=CHAVE_JOGO + ['_modelo'], keep='first'
    )

    pivot_prob = dados_janela.pivot(
        index=CHAVE_JOGO, columns='_modelo', values='Probabilidade Classe 1'
    )

    pivot_real = (
        dados_janela
        .drop_duplicates(subset=CHAVE_JOGO)
        .set_index(CHAVE_JOGO)['Resultado Real']
    )

    idx_comum  = pivot_prob.dropna().index
    pivot_prob = pivot_prob.loc[idx_comum]
    y_real     = pivot_real.loc[idx_comum]

    media_prob        = pivot_prob.mean(axis=1)
    previsao_ensemble = (media_prob >= 0.5).astype(int)

    temporadas_lista = [idx[0] for idx in idx_comum]
    df_resultado = pd.DataFrame({
        'Temporada':           temporadas_lista,
        'Media Probabilidade': media_prob.values,
        'Previsao Ensemble':   previsao_ensemble.values,
        'Resultado Real':      y_real.values,
    })

    for temporada, grupo in df_resultado.groupby('Temporada'):
        acuracia = (grupo['Previsao Ensemble'] == grupo['Resultado Real']).mean()
        f1       = f1_score(
            grupo['Resultado Real'],
            grupo['Previsao Ensemble'],
            average='weighted',
            zero_division=0
        )
        resultados.append({
            'Janela Incremental': janela,
            'Temporada':          temporada,
            'Acur\u00e1cia':           round(acuracia, 4),
            'F1-Score':           round(f1, 4),
        })

    print(f'Janela {janela} \u2014 jogos avaliados: {len(idx_comum)}')

df_soft = pd.DataFrame(resultados)
os.makedirs(SAIDA, exist_ok=True)
saida_path = os.path.join(SAIDA, 'ensemble_soft_voting.csv')
df_soft.to_csv(saida_path, index=False)

print(f'\nSalvo em: {saida_path}')
print(df_soft.to_string(index=False))

## Funcao de Plot

Criei a função `plot_linhas_com_soft_voting(janela)` que gera um grafico com duas partes:

- **Painel superior**: linhas de acuracia por temporada para cada modelo individual + a linha tracejada preta do soft voting
- **Painel inferior**: barras mostrando a quantidade de jogos avaliados por temporada

In [ ]:
def plot_linhas_com_soft_voting(janela_plot):
    base_path = os.path.abspath(os.path.join('..', 'results', 'experimento_03'))

    modelos = {
        'SVM':                'svm',
        'KNN':                'knn',
        'Extra Trees':        'extra_trees',
        'Regressao Logistica':'regressao_logistica',
        'LightGBM':           'lightgbm',
    }

    jogos_por_temporada = {
        '2008-2009': 237, '2009-2010': 162, '2011-2012': 176,
        '2012-2013': 350, '2013-2014': 316, '2014-2015': 285,
        '2015-2016': 255, '2016-2017': 260, '2018-2019': 218,
        '2019-2020': 207, '2020-2021': 266, '2021-2022': 304,
        '2022-2023': 312, '2023-2024': 386
    }

    cores = {
        'SVM':                '#1f77b4',
        'KNN':                '#ff7f0e',
        'Extra Trees':        '#2ca02c',
        'Regressao Logistica':'#d62728',
        'LightGBM':           '#9467bd',
    }

    marcadores = {
        'SVM':                'o',
        'KNN':                's',
        'Extra Trees':        '^',
        'Regressao Logistica':'D',
        'LightGBM':           'P',
    }

    temporadas_order = list(jogos_por_temporada.keys())

    dfs_plot = []
    for nome_modelo, arquivo_modelo in modelos.items():
        path = os.path.join(
            base_path,
            f'{arquivo_modelo}_experimento_03_janela{janela_plot}.csv'
        )
        if not os.path.exists(path):
            print(f'Arquivo nao encontrado: {path}')
            continue
        df_m = pd.read_csv(path)
        df_m['Modelo'] = nome_modelo
        dfs_plot.append(df_m)

    dados_plot = pd.concat(dfs_plot, ignore_index=True)
    dados_plot['Acur\u00e1cia'] = dados_plot['Acur\u00e1cia'].astype(float)
    dados_plot['Temporada'] = pd.Categorical(
        dados_plot['Temporada'], categories=temporadas_order, ordered=True
    )
    dados_plot = dados_plot.sort_values('Temporada')

    voto_path = os.path.join(base_path, 'ensemble_soft_voting.csv')
    df_voto = pd.read_csv(voto_path)
    df_voto = df_voto[df_voto['Janela Incremental'] == janela_plot].copy()
    df_voto['Acur\u00e1cia'] = df_voto['Acur\u00e1cia'].astype(float)
    df_voto['Temporada'] = pd.Categorical(
        df_voto['Temporada'], categories=temporadas_order, ordered=True
    )
    df_voto = df_voto.sort_values('Temporada')

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(16, 9),
        gridspec_kw={'height_ratios': [3, 1]},
        sharex=True
    )

    for nome_modelo in modelos.keys():
        subset = dados_plot[dados_plot['Modelo'] == nome_modelo].sort_values('Temporada')
        ax1.plot(
            subset['Temporada'],
            subset['Acur\u00e1cia'],
            marker=marcadores[nome_modelo],
            color=cores[nome_modelo],
            linewidth=2,
            markersize=6,
            label=nome_modelo,
            alpha=0.75
        )

    ax1.plot(
        df_voto['Temporada'],
        df_voto['Acur\u00e1cia'],
        marker='*',
        color='black',
        linewidth=2.5,
        markersize=10,
        linestyle='--',
        label='Soft Voting',
        zorder=5
    )

    ax1.set_title(
        f'Experimento 03 \u2014 Acuracia por temporada | Janela = {janela_plot} (soma)',
        fontsize=14
    )
    ax1.set_ylabel('Acuracia', fontsize=12)
    ax1.set_ylim(0.50, 0.85)
    ax1.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
    ax1.legend(loc='upper right', fontsize=10)
    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    ax1.tick_params(axis='x', rotation=45)

    temporadas_presentes = [t for t in temporadas_order if t in jogos_por_temporada]
    jogos_vals = [jogos_por_temporada[t] for t in temporadas_presentes]

    ax2.bar(
        temporadas_presentes, jogos_vals,
        color='#1f77b4', alpha=0.4, edgecolor='#1f77b4'
    )
    for i, v in enumerate(jogos_vals):
        ax2.text(i, v + 5, str(v), ha='center', va='bottom', fontsize=8)

    ax2.set_ylabel('Jogos', fontsize=10)
    ax2.set_ylim(0, max(jogos_vals) * 1.25)
    ax2.grid(axis='y', linestyle='--', alpha=0.3)
    ax2.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

## Visualizacao

Abaixo plotei o grafico comparativo para cada uma das 3 janelas incrementais (5, 10, 15).

Cada grafico mostra:
- As 5 linhas coloridas dos modelos individuais
- A linha preta tracejada de como o soma de votos se comportou
- Barras com a quantidade de jogos por temporada

In [ ]:
for janela in [5, 10, 15]:
    plot_linhas_com_soft_voting(janela)